<a href="https://colab.research.google.com/github/siva567-pixel/IntermediateAssessment-4/blob/main/Intermediate_assessment_3_jesna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting the Sex of Laysan Albatross



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

# Find the competition CSV files
data_folder = Path("/content/drive/MyDrive/DSA/Case study 4")
train_files = list(data_folder.rglob("train.csv"))
test_files = list(data_folder.rglob("test.csv"))

print("Train files:", train_files)
print("Test files:", test_files)

if not train_files or not test_files:
    raise FileNotFoundError("Add the competition dataset using Kaggle's Add Input button.")

train = pd.read_csv(train_files[0])
test = pd.read_csv(test_files[0])

print("Training data:", train.shape)
print("Test data:", test.shape)
train.head()

Train files: []
Test files: []


FileNotFoundError: Add the competition dataset using Kaggle's Add Input button.

## Check the data

In [ ]:
print(train["sexo"].value_counts())
print("\nMissing values:")
print(train.isnull().sum())

## Prepare and validate the model

In [ ]:
features = [col for col in train.columns if col not in ["id", "sexo"]]
X = train[features]
y = train["sexo"]
X_test = test[features]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SVM works better when the numeric features are scaled
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    SVC(C=10, kernel="rbf", probability=True, random_state=42)
)

model.fit(X_train, y_train)

m_index = list(model.classes_).index("M")
valid_prob = model.predict_proba(X_valid)[:, m_index]

# This threshold is an experiment; compare the leaderboard result
valid_pred = ["M" if p >= 0.40 else "H" for p in valid_prob]

print("Validation accuracy:", round(accuracy_score(y_valid, valid_pred), 3))
print("Validation macro F1:", round(f1_score(y_valid, valid_pred, average="macro"), 3))
print("Validation F1 for M:", round(f1_score(y_valid, valid_pred, pos_label="M"), 3))

## Train on all labeled data

In [ ]:
model.fit(X, y)

m_index = list(model.classes_).index("M")
test_prob = model.predict_proba(X_test)[:, m_index]
test_pred = ["M" if p >= 0.40 else "H" for p in test_prob]

print("Number of predictions:", len(test_pred))
print(pd.Series(test_pred).value_counts())

## Create the Kaggle submission

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "sexo": test_pred
})

submission.to_csv("/content/drive/MyDrive/DSA/Case study 4/1submission.csv", index=False)
print("Submission shape:", submission.shape)
submission.head()